# 11 — Полный batched Env → LLM → rollout → PPO loss pipeline

Эта тетрадка выполняет весь текущий synchronous MVP path: параллельные CrafText среды, MegaPrompts, один Tunix sampler call на batch в каждом decision round, batched environment steps/reset, per-env replay, token batches и masked PPO loss.

> Нужен явный local Qwen snapshot. Это проверка data-flow/loss contracts: actor logprobs и critic values пока не пересчитываются trainable RLCluster workload, поэтому loss ниже — smoke формы, не обучение Qwen.


In [ ]:
from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tunix_craftext.algorithms import masked_token_ppo_loss, masked_token_returns
from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.replay import save_replay
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.text_trajectory import text_trajectory_from_replay
from tunix_craftext.tunix_adapter import QwenTunixBackend


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None: raise RuntimeError('Run inside tunix-craftext.')
SNAPSHOT = ROOT / 'artifacts/models/qwen25-05b-instruct'
if not SNAPSHOT.is_dir(): raise FileNotFoundError(f'No explicit local snapshot: {SNAPSHOT}')
config = load_mvp_config(ROOT / 'configs/mvp/qwen_craftext.yaml')
runtime = build_craftext_runtime(config)
BATCH_SIZE, HORIZON, MAX_NEW_TOKENS = 2, 2, 8
print('actions:', runtime.actions.labels)


In [ ]:
rollout = collect_batched_text_rollout(
    runtime.adapter, MegaPromptRenderer(config.prompt.template),
    QwenTunixBackend(SNAPSHOT, cache_size=2048, seed=config.run.seed),
    actions=runtime.actions, batch_size=BATCH_SIZE, horizon=HORIZON, seed=config.run.seed,
    goal='Stay alive and inspect the world.', max_new_tokens=MAX_NEW_TOKENS,
    invalid_action='fallback', fallback_action_id=runtime.actions.index_of('NOOP'),
)
for t, (decision, reset_mask) in enumerate(zip(rollout.decisions, rollout.reset_after_step, strict=True)):
    print(f't={t}', 'actions=', [x.label for x in decision.actions], 'reward=', decision.transition.reward.tolist(), 'reset=', reset_mask.tolist())


In [ ]:
replays = replays_from_batched_rollout(
    rollout, config_path='configs/mvp/qwen_craftext.yaml', commit='notebook', backend='tunix-single-device:Qwen'
)
output_dir = ROOT / 'artifacts/trajectories/end-to-end-batched-qwen'
for env_index, replay in enumerate(replays):
    save_replay(output_dir / f'env-{env_index}.json', replay)
    print('env', env_index, 'steps=', len(replay.steps), 'fallbacks=', sum(s.fallback_used for s in replay.steps))


In [ ]:
# Each replay has time on its batch axis. Concatenate env trajectories only after inspecting them.
batches = tuple(text_trajectory_from_replay(replay) for replay in replays)
batch = batches[0]
print('one env token batch:', batch.token_ids.shape, 'prompt batch:', batch.prompt_token_ids.shape)
print('token mask:', batch.token_mask.tolist())
print('policy mask:', batch.policy_mask.tolist())
print('terminal rewards:', batch.rewards.tolist())


In [ ]:
returns = masked_token_returns(batch.rewards, batch.token_mask, gamma=0.99)
# Smoke-only: a real actor recomputes new_logprobs and critic supplies new/old values.
zero_values = jnp.zeros_like(batch.old_logprobs)
loss, metrics = masked_token_ppo_loss(
    batch.old_logprobs, batch.old_logprobs, returns, zero_values, zero_values, returns,
    batch.policy_mask, clip_epsilon=0.2, value_coefficient=0.5, entropy=zero_values, entropy_coefficient=0.01,
)
print('returns:', returns.tolist())
print('masked PPO loss:', float(loss))
print({name: float(value) for name, value in metrics.items()})


In [ ]:
actions = jnp.stack([jnp.asarray([item.action_id for item in decision.actions]) for decision in rollout.decisions])
rewards = jnp.stack([decision.transition.reward for decision in rollout.decisions])
figure, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(actions.T, aspect='auto', interpolation='nearest')
axes[0].set(title='actions [env, time]', xlabel='time', ylabel='env')
axes[1].imshow(rewards.T, aspect='auto', interpolation='nearest', cmap='RdYlGn')
axes[1].set(title='rewards [env, time]', xlabel='time', ylabel='env')
figure.tight_layout()
plt.show()


## Что остаётся до настоящего PPO update

1. RLCluster создаёт trainable actor, frozen reference и trainable critic на role meshes.
2. Actor пересчитывает `new_logprobs` на `token_ids`; critic выдаёт values.
3. Flashbax staging/rollout batch становится input Optax update.
4. Checkpoint сохраняет actor, critic, optimizer и policy version.
